# Session 9 — Validate against CelesTrak SOCRATES

**Phase 2.5 · trust the numbers.** SOCRATES (Satellite Orbital Conjunction Reports Assessing Threatening Encounters in Space) is CelesTrak's public conjunction service. We take a handful of its currently-reported conjunctions and check that OrbitGuard flags the same pairs with a comparable TCA and miss distance.

*The bar:* order-of-magnitude agreement. Our TLE snapshot may differ in epoch from theirs, so exact numbers won't match — but if we're within a few km / a few minutes on real events, the pipeline is trustworthy.

*Debug checklist if numbers are off:* (1) UTC vs local time, (2) TLE epoch mismatch, (3) frame consistency (both objects in the same inertial frame).

In [ ]:
import sys; sys.path.insert(0, '../src')
from orbitguard import pipeline, report

# SOCRATES top-conjunction feeds (public):
#   https://celestrak.org/SOCRATES/  (HTML)
#   https://celestrak.org/NORAD/elements/gp.php  (GP data by CATNR)
# Pick a SOCRATES row: note its two NORAD ids, TCA, and reported miss distance.
SOCRATES_EXAMPLE = {
    'norad_a': 57154, 'norad_b': 67112,   # <- replace with a current SOCRATES pair
    'reported_tca_utc': '2026-07-30 20:19',
    'reported_miss_km': 0.5,
}
SOCRATES_EXAMPLE

In [ ]:
# Run OrbitGuard over the same window and pull the matching pair out of the report.
res = pipeline.run(group='active', hours=24, threshold_km=10, verbose=False)
df = report.to_dataframe(res.ranked)

a, b = SOCRATES_EXAMPLE['norad_a'], SOCRATES_EXAMPLE['norad_b']
match = df[((df.norad_a == a) & (df.norad_b == b)) | ((df.norad_a == b) & (df.norad_b == a))]
print('SOCRATES says:', SOCRATES_EXAMPLE)
print('\nOrbitGuard says:')
print(match[['object_a','object_b','tca_utc','miss_km','rel_speed_kms','risk_score']].to_string(index=False))

**Done when:** for a few SOCRATES events, OrbitGuard flags the same pair with a miss distance and TCA in the same ballpark. Record the comparison here as evidence.

> Note: the example NORAD ids above are seeded from our own top event so the notebook runs; swap in live SOCRATES rows for a true independent check.